# Deep Network Development, 2026 Autumn - Week 5 Practice - **Notebook for video recording**

## Managing a clothing chain

**You manage an international luxury clothing chain which has shops in several cities worldwide. The clothing chain sells haute couture clothing items produced from various, locally sourced grasses and shrubs. You are tasked to perform some calculations regarding the profitability of different products and shops of the company.**

The chain has four shops worldwide and each of them is selling the same three types of products. The following are given:
- Names of the products: `product_names_list`
- Names of the shop locations : `shop_names_list`
- The number of products sold in each shop for each product type in a given month: `num_sold_products`
- The prices of products (Each product type has a similar price in all shops): `product_prices`
- Profit margin of shops (prices are the same in all the shops, but for some locations it is cheaper to source the products): `profit_margin_of_shops`
- The static cost of running the shops for a month: `cost_of_shops`
- The static cost of running the chain / headquarters for a month beyond the shop costs: a scalar: `cost_of_hq`


In [ ]:
import numpy as np

In [ ]:

product_names_list = ["Swimming suit", "Raincoat", "Sweater"]          # list with length: n_product_types
shop_names_list = ["Hanoi", "Reykjavík", "Tunis", "Ulaanbaatar"]       # list with length: n_shops

num_sold_products = np.array([[373, 711,  49 ],
                              [ 75, 182,  207],
                              [196,  17,  24 ],
                              [  7,   3,  96 ]], dtype=np.int32)                     # (n_shops, n_product_types) of int

product_prices = np.array([800., 2250., 1900.], dtype=np.float32)                    # (n_product_types,) of float

profit_margin_of_shops = np.array([0.45, 0.17, 0.52, 0.78], dtype=np.float32)        # (n_shops,) of float

cost_of_shops = np.array([91_250., 275_000., 78_000., 45_700.], dtype=np.float32)    # (n_shops,) of float

cost_of_hq = 145_000.




### Best selling product of each shop


**Task:** For each shop, return the products which are sold in the highest number. Instead of indices, return the names of the shops with the best selling product names. Return the number of items sold from these products in the given shops alongside with the names. Return the result in the `best_selling_products_per_shop` array shaped `(n_shops, 3)`.

**Example result:**
```
array([["Hanoi", "Raincoat", 711],
       ["Reykjavík", "Sweater", 207],
       ["Tunis", "Swimming suit", 196],
       ["Ulaanbaatar", "Sweater", 96]])
```


In [ ]:

product_names_arr = np.array(product_names_list, dtype=object)           # Note: dtype=object is not efficient, but can be used to store any Python objects as elements; specifically for strings, there are other, more efficient dtypes ('<U*', StringDType, ...)
shop_names_arr = np.array(shop_names_list, dtype=object)
n_shops = shop_names_arr.shape[0]

best_selling_prod_idxs = np.argmax(num_sold_products, axis=1)            # (n_shops, n_product_types) -> (n_shops,)

best_selling_prod_numbers = num_sold_products[ np.arange(n_shops), best_selling_prod_idxs ]       # (n_shops,) int64
#best_selling_prod_numbers = np.amax(num_sold_products, axis=1)          # a simpler option in this case...

best_selling_prod_names = product_names_arr[best_selling_prod_idxs]      # (n_shops,) object

best_selling_products_per_shop = np.stack([shop_names_arr, best_selling_prod_names, best_selling_prod_numbers], axis=1)    # (n_shops,), (n_shops,), (n_shops,) -> (n_shops, 3)



print(best_selling_products_per_shop)


[['Hanoi' 'Raincoat' 711]
 ['Reykjavík' 'Sweater' 207]
 ['Tunis' 'Swimming suit' 196]
 ['Ulaanbaatar' 'Sweater' 96]]


### Total profit (v1)

**Task:** Calculate the total profit (a scalar float) and assign it to the `total_profit` variable. The total profit is given by the following formulas:

$\displaystyle \textrm{totalProfit} = \sum^{\textrm{nShops}-1}_{\textrm{shopIdx} = 0}\Bigl[\ \textrm{grossProfitPerShop}[\ \textrm{shopIdx}\ ] - \textrm{costOfShops}[\ \textrm{shopIdx}\ ]\ \Bigr]\ - \textrm{costOfHq}$

$\displaystyle \textrm{grossProfitPerShop}[\ \textrm{shopIdx}\ ] = \sum^{\textrm{nProducts}-1}_{\textrm{prodIdx} = 0}\Bigl[\ \textrm{numProductsSold}[\ \textrm{shopIdx},\ \textrm{prodIdx}\ ] \cdot \textrm{productPrices}[\ \textrm{prodIdx}\ ] \cdot  \\ \qquad\qquad\qquad
\cdot \textrm{profitMarginOfShops}[\ \textrm{shopIdx}\ ]\ \Bigr]$


In [ ]:


revenues = num_sold_products * product_prices                       # (n_shops, n_product_types)   v (n_product_types,)   -> (n_shops, n_product_types)

revenues_T = revenues.T                                             # (n_shops, n_product_types)                          -> (n_product_types, n_shops)

gross_profits = revenues_T * profit_margin_of_shops                 # (n_product_types, n_shops)   v (n_shops,)           -> (n_product_types, n_shops)

gross_profit_per_shop = np.sum(gross_profits, axis=0)               # (n_product_types, n_shops)                          -> (n_shops,)

profit_per_shop = gross_profit_per_shop - cost_of_shops             # (n_shops,)                   v (n_shops,)           -> (n_shops,)

total_profit_no_hq = np.sum(profit_per_shop, axis=0)                # (n_shops,)                                          -> scalar

total_profit = total_profit_no_hq - cost_of_hq                      # scalar                       v scalar               -> scalar



#    Same in one line:
# total_profit = np.sum(np.sum((num_sold_products * product_prices).T * profit_margin_of_shops, axis=0) - cost_of_shops, axis=0) - cost_of_hq

print(total_profit)


684831.4676433802


### Total profit (v2)

(Same task as before)

In [ ]:

revenues = num_sold_products * product_prices                       # (n_shops, n_product_types)   v (n_product_types,)   -> (n_shops, n_product_types)

profit_margin_of_shops_2d = profit_margin_of_shops.reshape(-1, 1)   # (n_shops,)                                          -> (n_shops, 1)
#profit_margin_of_shops_2d = profit_margin_of_shops_2d[:, None]     # (n_shops,)                                          -> (n_shops, 1), same as above

gross_profits = revenues * profit_margin_of_shops_2d                # (n_shops, n_product_types)   v (n_shops, 1)        -> (n_shops, n_product_types)

gross_profit_per_shop = np.sum(gross_profits, axis=1)               # (n_shops, n_product_types)                          -> (n_shops,)

profit_per_shop = gross_profit_per_shop - cost_of_shops             # (n_shops,)                   v (n_shops,)           -> (n_shops,)

total_profit_no_hq = np.sum(profit_per_shop, axis=0)                # (n_shops,)                                          -> scalar

total_profit = total_profit_no_hq - cost_of_hq                      # scalar                       v scalar               -> scalar



#    Same in one line:
# total_profit = np.sum(np.sum(num_sold_products * product_prices * profit_margin_of_shops[:, None], axis=1) - cost_of_shops, axis=0) - cost_of_hq

print(total_profit)


684831.4676433802


### Number of products to sell for break-even

**Task:** Compute the number of products from each product type needed to be sold in order to cover the costs of running each shop. Return the integer number (ceiling) of products needed for each product type for each shop. The result should be an array of shape `(n_shops, n_product_types)` with an integer datatype, assigned to the `num_products_to_cover_shop_cost_i` variable.

**Example result:**
```
array([[  254,  91, 107],
       [ 2023, 719, 852],
       [  188,  67,  79],
       [   74,  27,  31]])
```

In [ ]:

# as the formula to calculate this is very simple, the cheapest location and most expensive product combo will be the winner, but let's not assume this, find this in the more general, naive way!

gross_profits_per_product_shop = product_prices * profit_margin_of_shops[:, None]            # (n_product_types,)    v (n_shops, 1)                 -> (n_shops, n_product_types)

num_products_to_cover_shop_cost = cost_of_shops[:, None] / gross_profits_per_product_shop    # (n_shops, 1)          v (n_shops, n_product_types)   -> (n_shops, n_product_types)

num_products_to_cover_shop_cost_i = np.ceil(num_products_to_cover_shop_cost).astype(np.int64)     # (n_shops, n_product_types) float -> (n_shops, n_product_types) int



#    Same in one line:
# num_products_to_cover_shop_cost_i = np.ceil(cost_of_shops[:, None] / product_prices * profit_margin_of_shops[:, None]).astype(np.int64)

print(num_products_to_cover_shop_cost_i)


[[ 254   91  107]
 [2023  719  852]
 [ 188   67   79]
 [  74   27   31]]


### Very profitable products

**Task:** Return the result as the array `very_profitable_product_shop_pairs` of shape `(k, 3)` where those shop-product pairs are listed where less than 100 items from a product in the specific shop can fully support the running costs of that shop. Each row of the array will be a (shop_name, product_name, number_of_products) triple.

**Example result:**
```
array([['Hanoi', 'Raincoat', 91],
       ['Tunis', 'Raincoat', 67],
       ['Tunis', 'Sweater', 79],
       ['Ulaanbaatar', 'Swimming suit', 74],
       ['Ulaanbaatar', 'Raincoat', 27],
       ['Ulaanbaatar', 'Sweater', 31]])
```


In [ ]:

product_names_arr = np.array(product_names_list, dtype=object)           # Note: dtype=object is not efficient, but can be used to store any Python objects as elements; specifically for strings, there are other, more efficient dtypes ('<U*', StringDType, ...)
shop_names_arr = np.array(shop_names_list, dtype=object)



mask_where_condition = num_products_to_cover_shop_cost_i < 100      # (n_shops, n_product_types) int < scalar int -> (n_shops, n_product_types) bool_

idxs_tup_where_condition = np.where(mask_where_condition)           # tuple: ( array with shape (n_idxs_with_condition,) , array with shape (n_idxs_with_condition,) )
idxs0_where_condition, idxs1_where_condition = idxs_tup_where_condition     # unpack tuple

shop_names_where_condition = shop_names_arr[idxs0_where_condition]
product_names_where_condition = product_names_arr[idxs1_where_condition]

num_products = num_products_to_cover_shop_cost_i[mask_where_condition]
#num_products = num_products_to_cover_shop_cost_i[idxs_tup_where_condition]      # also good: indexing with the n-dimensional indices (with a tuple of n pieces of integer arrays)

very_profitable_product_shop_pairs = np.stack([shop_names_where_condition, product_names_where_condition, num_products], axis=1)



print(very_profitable_product_shop_pairs)


[['Hanoi' 'Raincoat' 91]
 ['Tunis' 'Raincoat' 67]
 ['Tunis' 'Sweater' 79]
 ['Ulaanbaatar' 'Swimming suit' 74]
 ['Ulaanbaatar' 'Raincoat' 27]
 ['Ulaanbaatar' 'Sweater' 31]]


### One of the very profitable products

**Task:** Return a single such (shop_name, product_name) pair as a tuple of two strings and assign the tuple to the `one_very_profitable_shop_product_pair`.


**Example result:**
```
('Hanoi', 'Raincoat')
```

In [ ]:

first_flat_idx_with_condition = np.argmax(mask_where_condition)      # a single flat index as an int

first_idx_with_condition = np.unravel_index(first_flat_idx_with_condition, shape=mask_where_condition.shape)   # a single multi-dimensional index as a tuple of ints

one_very_profitable_shop_product_pair = ( shop_names_arr[first_idx_with_condition[0]], product_names_arr[first_idx_with_condition[1]] )


print(one_very_profitable_shop_product_pair)


('Hanoi', 'Raincoat')
